# Kahn's Algorithm — Demo

Demonstrates the schedulers in `kahns/` (serial, parallel, parallel-efficient, DFS) and a quick profiling comparison between the two parallel implementations.

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Colab starts with an empty runtime, so clone the repo first.
    REPO_URL = "https://github.com/jhell1717/arm_code.git"
    REPO_DIR = Path("/content/arm_code")
    if not REPO_DIR.exists():
        !git clone --depth 1 {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)

print("Working directory:", os.getcwd())

In [12]:
import sys
from pathlib import Path

# notebooks/ is a sibling of kahns/, so add the project root to sys.path
# so `kahns` resolves as a package.
ROOT = Path.cwd().parent if (Path.cwd() / "kahns").exists() is False else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ROOT

PosixPath('/Users/joshuahellewell/Desktop/04-career/ARM/arm_code')

In [13]:
from kahns import (
    Task,
    make_serial_schedule,
    make_parallel_stages,
    make_parallel_stages_eff,
    make_dfs_schedule,
)

## Scheduling demo

Same example task graph used in `kahns/main.py`.

In [11]:
tasks = [
    Task("package", ["link"]),
    Task("compile_b", ["parse"]),
    Task("compile_a", ["parse"]),
    Task("link", ["compile_a", "compile_b"]),
    Task("parse", []),
]

print("Serial:           ", make_serial_schedule(tasks))
print("Parallel:         ", make_parallel_stages(tasks))
print("Parallel (eff):   ", make_parallel_stages_eff(tasks))
print("DFS:              ", make_dfs_schedule(tasks))

Serial:            ['parse', 'compile_a', 'compile_b', 'link', 'package']
Parallel:          [['parse'], ['compile_a', 'compile_b'], ['link'], ['package']]
Parallel (eff):    [['parse'], ['compile_b', 'compile_a'], ['link'], ['package']]
DFS:               ['parse', 'compile_a', 'compile_b', 'link', 'package']


## Profiling: `make_parallel_stages` vs `make_parallel_stages_eff`

Reuses the helpers from `kahns/profile_parallel.py` to build a larger synthetic DAG (many layers of independent tasks) where the algorithmic difference actually shows up.

In [5]:
from kahns.profile_parallel import make_layered_tasks, bench

big_tasks = make_layered_tasks(num_layers=50, width=20)  # 1000 tasks
number = 100

naive_time = bench(make_parallel_stages, big_tasks, number)
eff_time = bench(make_parallel_stages_eff, big_tasks, number)

print(f"make_parallel_stages:     {naive_time / number * 1e6:.2f} us/call")
print(f"make_parallel_stages_eff: {eff_time / number * 1e6:.2f} us/call")
print(f"speedup: {naive_time / eff_time:.2f}x")

make_parallel_stages:     3696.78 us/call
make_parallel_stages_eff: 1621.27 us/call
speedup: 2.28x


### Notes:
* For small graphs, the gains in using the optimisations vs the original parallel implementation are minimal.
* This is a result of the redundant work being done at each stage, by checking through all the tasks and dependencies regardless of how relevant they are at the time. 